# Lesson 4.3 - CI/CD and Secrets

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Create a CI/CD workflow that runs PowerShell checks
- Handle secrets without placing them in plain text
- Connect local PowerShell scripts to a pipeline-friendly layout

**Project file(s) produced:** `.github/workflows/powershell-ci.yml` and `scripts/Use-SecureSecret.ps1`

## Concepts

CI/CD stands for **continuous integration** and **continuous delivery/deployment**. In practice, that means scripts are checked automatically whenever code changes, instead of only being tested by hand.

Secrets should stay out of source control. The common pattern is to read them from the environment, a secret store, or a pipeline variable group and convert them to a secure object only when needed.

Reference: [PowerShell Documentation](https://learn.microsoft.com/en-us/powershell/)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap

lesson_dir = Path('study-plan/topic-powershell/phase-4-cloud-devops')
workflow_dir = lesson_dir / '.github/workflows'
script_dir = lesson_dir / 'scripts'
workflow_dir.mkdir(parents=True, exist_ok=True)
script_dir.mkdir(parents=True, exist_ok=True)

def run_powershell(script_text: str, script_name: str) -> int:
    script_path = lesson_dir / script_name
    script_path.write_text(script_text, encoding='utf-8')
    executable = shutil.which('pwsh') or shutil.which('powershell')
    if executable:
        result = subprocess.run([executable, '-NoLogo', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(script_path)], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        return result.returncode
    print(f'{script_path.name} written. Run it in PowerShell to see the live output.')
    return 0

## Code-Along: Step 1 - Add a pipeline workflow

This workflow installs the tools needed for checks and then runs the PowerShell validation steps. It is intentionally small so you can see the structure clearly.

In [ ]:
workflow_path = workflow_dir / 'powershell-ci.yml'
workflow_content = textwrap.dedent(r'''
name: PowerShell CI

on:
  push:
  pull_request:

jobs:
  validate:
    runs-on: windows-latest
    steps:
      - uses: actions/checkout@v4
      - name: Install PowerShell modules
        shell: pwsh
        run: |
          Install-Module PSScriptAnalyzer -Scope CurrentUser -Force
          Install-Module Pester -Scope CurrentUser -Force
      - name: Run analyzer
        shell: pwsh
        run: Invoke-ScriptAnalyzer -Path . -Recurse
      - name: Run tests
        shell: pwsh
        run: Invoke-Pester -Path ./tests -CI
''')
workflow_path.write_text(workflow_content, encoding='utf-8')
print(workflow_path.read_text(encoding='utf-8'))

## Code-Along: Step 2 - Handle secrets safely

This script shows the shape of secure credential handling. The important part is that the secret is not hard-coded into the script itself.

In [ ]:
script_path = script_dir / 'Use-SecureSecret.ps1'
script_content = textwrap.dedent(r'''
param(
    [Parameter(Mandatory)]
    [string]$Username
)

$plainText = $env:DEMO_PASSWORD
if (-not $plainText) {
    Write-Host 'No secret was provided in the environment.'
    return
}

$securePassword = ConvertTo-SecureString $plainText -AsPlainText -Force
$credential = [pscredential]::new($Username, $securePassword)
Write-Host "Created a credential object for $($credential.UserName) without echoing the password."
''')
script_path.write_text(script_content, encoding='utf-8')
print(script_path.read_text(encoding='utf-8'))

## Exercise

Add one more pipeline step that caches or publishes an artifact created by one of the earlier lessons.

**Your task:**
1. Add one step to the YAML workflow.
2. Keep the workflow readable.
3. Confirm the pipeline still includes install, test, and analyzer steps.

**Expected result:** a CI/CD workflow that validates PowerShell work automatically.

In [ ]:
# TODO: add an artifact publishing or caching step to the workflow
exercise_script = textwrap.dedent(r'''
Write-Host "TODO: extend the pipeline"
''')
run_powershell(exercise_script, 'lesson-4.3-exercise.ps1')

## Key Takeaways

- CI/CD makes PowerShell checks repeatable and visible to the team.
- Secrets should be read from the environment or a secret store, not typed into the script.
- A small workflow is enough to start enforcing quality automatically.

---

**Next:** [Capstone Project](../capstone/capstone-project.ipynb)